# Phase 4 — Graph Analysis (NetworkX)
**Marina Osmolovska — MaCAD S3 Graph ML Final**

Converts the TopologicPy access graph `g2` to NetworkX and computes:
- **depth-from-entrance** — drives slab heights in Phase 5 (design)
- betweenness, closeness, degree centrality

**Run Phase 3 notebook first in the same kernel session** so that `g2`, `ROOM_COLOR`, and `ROOM_LABEL` are already in scope.
If you started a fresh kernel, run the *Quick rebuild* cell below first.

## 1. Imports

In [ ]:
import networkx as nx
from topologicpy.Graph import Graph
from topologicpy.Edge import Edge
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

## 2. Quick rebuild (skip if Phase 3 is already in scope)

Uncomment and run this cell only if `g2` is not defined (fresh kernel).

In [ ]:
# Uncomment the line below to rebuild g2 from Phase 3 in a fresh kernel.
# %run "MO_Final_Phase3_AccessGraph.ipynb"

## 3. Convert TopologicPy graph → NetworkX

Tries the built-in `Graph.NetworkXGraph` API first.
Falls back to a manual build from `Graph.Vertices` / `Graph.Edges` if the API is not available.

In [ ]:
if hasattr(Graph, "NetworkXGraph"):
    nxGraph = Graph.NetworkXGraph(g2)
    print("Used Graph.NetworkXGraph")
else:
    nxGraph = nx.Graph()
    vertices = Graph.Vertices(g2)
    for i, v in enumerate(vertices):
        d = Topology.Dictionary(v)
        nxGraph.add_node(i,
            room_type = Dictionary.ValueAtKey(d, "room_type") or "unknown",
            color     = Dictionary.ValueAtKey(d, "color") or "grey")
    edges_list = Graph.Edges(g2)
    for e in edges_list:
        sv = Edge.StartVertex(e)
        ev = Edge.EndVertex(e)
        si = next((i for i, v in enumerate(vertices) if Vertex.Distance(sv, v) < 0.001), None)
        ei = next((i for i, v in enumerate(vertices) if Vertex.Distance(ev, v) < 0.001), None)
        if si is not None and ei is not None:
            de = Topology.Dictionary(e)
            nxGraph.add_edge(si, ei, door_type=Dictionary.ValueAtKey(de, "door_type") or "door")
    print("Built NetworkX graph manually from Graph.Vertices / Graph.Edges")

print("Nodes:", nxGraph.number_of_nodes(), "| Edges:", nxGraph.number_of_edges())

## 4. Find entrance node

Looks for the staircase/corridor node attached to an `entrance_door` edge.
This is the source for all depth calculations.

In [ ]:
entrance_node = None
for u, v, data in nxGraph.edges(data=True):
    if data.get("door_type") == "entrance_door":
        for n in [u, v]:
            if nxGraph.nodes[n].get("room_type") in ("stairs", "corridor"):
                entrance_node = n
                break
    if entrance_node is not None:
        break

if entrance_node is None:
    for n, data in nxGraph.nodes(data=True):
        if data.get("room_type") in ("stairs", "corridor"):
            entrance_node = n
            print(f"WARNING: no entrance_door edge found — using first stairs/corridor node ({n}) as entrance")
            break

print("Entrance node:", entrance_node, "| room_type:", nxGraph.nodes[entrance_node].get("room_type"))

## 5. Depth from entrance (the design driver)

Unweighted hop-count from the entrance node to every other node.
This is what gets mapped to `slab_Z = -k × depth` in Phase 5.

In [ ]:
depth = nx.shortest_path_length(nxGraph, source=entrance_node)
nx.set_node_attributes(nxGraph, depth, "depth_from_entrance")
print("depth_from_entrance per node:")
print(depth)

## 6. Centrality metrics

In [ ]:
betweenness = nx.betweenness_centrality(nxGraph)
closeness   = nx.closeness_centrality(nxGraph)
degree_c    = nx.degree_centrality(nxGraph)

## 7. Results table

Sorted by depth. Copy the `depth` column — these values drive slab heights in Rhino/GH.

In [ ]:
print(f"{'node':<6} {'room_type':<14} {'depth':>5} {'betweenness':>12} {'closeness':>10} {'degree':>8}")
print("-" * 62)
for n, data in sorted(nxGraph.nodes(data=True), key=lambda x: depth.get(x[0], 99)):
    print(
        f"{n:<6} {data.get('room_type','?'):<14}"
        f"{depth.get(n,-1):>5}"
        f"{betweenness[n]:>13.4f}"
        f"{closeness[n]:>11.4f}"
        f"{degree_c[n]:>9.4f}"
    )